# RecFlowTime — evaluation

Scores generated samples with every metric reported in the paper. Run
`01_train_and_sample.ipynb` first, or point `RESULTS_DIR` at any directory
holding `<dataset>_<tag>_data.npz` files.

The metrics fall into two groups, and the distinction matters when reading
small differences.

**Learned metrics** train a network on the samples, so they carry estimator
variance even when the samples are fixed. We measured the discriminative
score's spread at 0.02–0.11 across evaluator seeds on fixed data, which is
larger than many published gaps between methods. Report them with error bars
over seeds.

| metric | what it does | lower is better |
|---|---|---|
| Discriminative | trains a GRU to separate real from generated; reports \|accuracy − 0.5\| | yes |
| Predictive | trains a GRU on generated data to forecast real data | yes |
| Context-FID | Fréchet distance in a TS2Vec representation | yes |

**Deterministic metrics** are closed-form functions of the samples. They carry
no evaluator variance, so differences in them are exact.

| metric | what it does | lower is better |
|---|---|---|
| Correlational | difference between feature cross-correlation matrices | yes |
| Spectral | mean absolute difference between average log power spectra | yes |
| Increment | 1-Wasserstein distance between step-to-step change distributions | yes |
| Autocorrelation | mean absolute difference between average ACFs | yes |

## 1. Settings

In [ ]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath("."))
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import torch
import matplotlib.pyplot as plt

RESULTS_DIR = os.path.join("results", "notebook")
DATASET     = "sines"     # must match a <DATASET>_<TAG>_data.npz in RESULTS_DIR
TAG         = "ours"

# These are the reported evaluation settings, identical to
#   scripts/evaluate_saved.py --repeat 2
# which produced every number in the paper: two evaluator seeds, two encoder
# fits, 800 TS2Vec steps, 1200 training steps for the discriminative and
# predictive networks. Two seeds bound the spread without estimating it
# precisely, so treat differences smaller than the spread as inconclusive.
#
# Raising N_SEEDS is cheap; raising N_CFID_SEEDS is not, because each Context-FID
# seed refits the TS2Vec encoder and that fit dominates the runtime.
N_SEEDS       = 2
N_CFID_SEEDS  = 2
TS2VEC_STEPS  = 800
METRIC_STEPS  = 1200

## 2. Load the samples

In [ ]:
path = os.path.join(RESULTS_DIR, f"{DATASET}_{TAG}_data.npz")
d = np.load(path)

real  = torch.from_numpy(d["real_test"]).float()
fake  = torch.from_numpy(d["generated"]).float()[: len(real)]
train = torch.from_numpy(d["real_train"]).float()

print(f"{path}")
print(f"  real  {tuple(real.shape)}")
print(f"  fake  {tuple(fake.shape)}")
print(f"  train {tuple(train.shape)}  (used to fit the Context-FID encoder)")

## 3. Deterministic metrics

These run in about a second and carry no evaluator variance.

In [ ]:
from recflowtime.metrics import (correlation_score, spectral_distance,
                                  delta_distance, acf_distance)

deterministic = {
    "Correlational":   correlation_score(real, fake),
    "Spectral":        spectral_distance(real, fake),
    "Increment":       delta_distance(real, fake),
    "Autocorrelation": acf_distance(real, fake),
}
for k, v in deterministic.items():
    print(f"  {k:18s} {v:.4f}")

## 4. Learned metrics

Each seed trains a fresh network. Context-FID additionally refits the TS2Vec
encoder per seed, which dominates the runtime — expect a few minutes per seed
rather than a few seconds.

In [ ]:
from recflowtime.metrics import (discriminative_score, predictive_score,
                                  fit_context_encoder, context_fid)

t0 = time.time()
disc = [discriminative_score(real, fake, steps=METRIC_STEPS, device="cpu", seed=s)
        for s in range(N_SEEDS)]
pred = [predictive_score(real, fake, steps=METRIC_STEPS, device="cpu", seed=s)
        for s in range(N_SEEDS)]
print(f"discriminative and predictive: {time.time() - t0:.0f}s")

t0 = time.time()
cfid = []
for s in range(N_CFID_SEEDS):
    enc = fit_context_encoder(train, device="cpu", steps=TS2VEC_STEPS, seed=s)
    cfid.append(context_fid(real, fake, enc, device="cpu"))
print(f"Context-FID ({N_CFID_SEEDS} encoder fits): {time.time() - t0:.0f}s")

learned = {"Discriminative": disc, "Predictive": pred, "Context-FID": cfid}

## 5. Results

In [ ]:
print(f"{DATASET}  ({TAG})\n" + "=" * 46)
print("learned (mean +- std over seeds; lower is better)")
for name, vals in learned.items():
    print(f"  {name:18s} {np.mean(vals):.4f} +- {np.std(vals):.4f}   "
          f"n={len(vals)}")
print("\ndeterministic (exact; lower is better)")
for name, v in deterministic.items():
    print(f"  {name:18s} {v:.4f}")

summary = {"dataset": DATASET, "tag": TAG,
           "learned": {k: {"mean": float(np.mean(v)), "std": float(np.std(v)),
                           "values": [float(x) for x in v], "n_seeds": len(v)}
                       for k, v in learned.items()},
           "deterministic": {k: float(v) for k, v in deterministic.items()}}
out = os.path.join(RESULTS_DIR, f"metrics_{DATASET}_{TAG}.json")
json.dump(summary, open(out, "w"), indent=2)
print(f"\nwrote {out}")

## 6. Visual checks

PCA and t-SNE of real against generated sequences, plus the average power
spectrum. Agreement here is necessary but not sufficient: a model can match
these projections while still producing locally implausible sequences, which is
what the spectral and increment distances are for.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

N_VIS = min(1000, len(real))
R = real[:N_VIS].reshape(N_VIS, -1).numpy()
F = fake[:N_VIS].reshape(N_VIS, -1).numpy()
REAL_C, GEN_C = "#E03131", "#1C7ED6"

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

p = PCA(n_components=2).fit(R)
for arr, c, lab in [(R, REAL_C, "real"), (F, GEN_C, "generated")]:
    z = p.transform(arr)
    ax[0].scatter(z[:, 0], z[:, 1], s=6, alpha=0.45, c=c, label=lab)
ax[0].set_title("PCA", fontsize=13, fontweight="bold"); ax[0].legend()

z = TSNE(n_components=2, perplexity=30, init="pca",
         random_state=0).fit_transform(np.concatenate([R, F]))
ax[1].scatter(z[:N_VIS, 0], z[:N_VIS, 1], s=6, alpha=0.45, c=REAL_C, label="real")
ax[1].scatter(z[N_VIS:, 0], z[N_VIS:, 1], s=6, alpha=0.45, c=GEN_C, label="generated")
ax[1].set_title("t-SNE", fontsize=13, fontweight="bold"); ax[1].legend()

for arr, c, lab in [(real, REAL_C, "real"), (fake, GEN_C, "generated")]:
    ps = np.abs(np.fft.rfft(arr.numpy(), axis=1)) ** 2
    ax[2].plot(np.log(ps.mean((0, 2)) + 1e-8), color=c, lw=2, label=lab)
ax[2].set_title("mean log power spectrum", fontsize=13, fontweight="bold")
ax[2].set_xlabel("frequency bin"); ax[2].legend()

for a in ax[:2]:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 7. Comparing configurations

To compare arms — for example the transport coupling against independent
pairing — train each in `01_train_and_sample.ipynb` under a different `TAG`,
then evaluate each here and compare.

Two cautions when reading the result. A difference in a learned metric smaller
than the spread across seeds is not evidence of a difference between models.
And these error bars cover evaluator variance only, not variance across
training seeds, so they do not establish that a ranking is stable across
initialisations.

In [ ]:
# Example: score every tag present for this dataset and tabulate.
import glob, re

rows = []
for f in sorted(glob.glob(os.path.join(RESULTS_DIR, f"metrics_{DATASET}_*.json"))):
    m = json.load(open(f))
    rows.append((m["tag"],
                 m["learned"]["Context-FID"]["mean"],
                 m["learned"]["Discriminative"]["mean"],
                 m["deterministic"]["Spectral"]))

if rows:
    print(f"{'tag':14s}{'Context-FID':>13}{'Discrim.':>11}{'Spectral':>11}")
    for t, c, di, sp in rows:
        print(f"{t:14s}{c:13.4f}{di:11.4f}{sp:11.4f}")
else:
    print("No metrics files yet — run section 5 for at least one tag.")